# VnExpress Step 1 - crawler smoke test

Notebook thử nghiệm tối thiểu cho **Data Collection MVP**:

1. Mở trang bằng Selenium và lấy rendered HTML.
2. Parse các trường cơ bản bằng BeautifulSoup.
3. Crawl mặc định tối đa **1 bài**.
4. Lưu raw HTML dạng `.html.gz` và metadata JSON vào `crawl_data/`.

Notebook này chưa làm deduplication nâng cao, NLP/LLM, embedding, RAG hay video. Hãy bảo đảm việc crawl phù hợp với robots.txt, điều khoản sử dụng và giới hạn truy cập của nguồn. Chrome/Chromium cần được cài trên máy để Selenium khởi động browser.

## Tai va cai dat thu vien

In [1]:
%pip install selenium beautifulsoup4 lxml

  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.5 MB ? eta -:--:--
   ------------------- -------------------- 4.7/9.5 MB 17.8 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 22.8 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)

   ------------- -------------------------- 2/6 [outcome]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   -------------------- ------------------- 3/6 [trio]
   --------------------------------- ----

In [10]:
from __future__ import annotations

import gzip
import hashlib
import json
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urljoin, urlsplit, urlunsplit

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

HOME_URL = "https://vnexpress.net/"
# Có thể dán trực tiếp URL bài viết vào đây để bỏ qua bước discovery.
TARGET_ARTICLE_URL = "https://vnexpress.net/ap-thap-nhiet-doi-manh-len-thanh-bao-5111866.html"
MAX_DISCOVERED_URLS = 5
MAX_ARTICLES_TO_CRAWL = 1
PAGE_TIMEOUT_SECONDS = 20
USER_AGENT = "AI-Tech-News-Research-Crawler/0.1"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_ROOT = PROJECT_ROOT / "crawl_data"
print(f"Project root: {PROJECT_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")

Project root: d:\sumary_newpapers
Output root: d:\sumary_newpapers\crawl_data


In [3]:
TRACKING_PARAMS = {
    "utm_source",
    "utm_medium",
    "utm_campaign",
    "utm_term",
    "utm_content",
    "fbclid",
    "gclid",
}


def normalize_url(url: str) -> str:
    """Remove tracking parameters while preserving useful query parameters."""
    parts = urlsplit(url.strip())
    clean_query = [
        (key, value)
        for key, value in parse_qsl(parts.query, keep_blank_values=True)
        if key.lower() not in TRACKING_PARAMS
    ]
    return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(clean_query), ""))


def url_hash(url: str) -> str:
    return hashlib.sha256(normalize_url(url).encode("utf-8")).hexdigest()[:16]


def clean_text(value: str | None) -> str | None:
    if not value:
        return None
    value = re.sub(r"\s+", " ", value).strip()
    return value or None

In [4]:
class SeleniumFetcher:
    """Fetch rendered HTML; parsing and storage stay outside this class."""

    def __init__(self, timeout: int = PAGE_TIMEOUT_SECONDS, headless: bool = True):
        options = Options()
        if headless:
            options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--window-size=1920,1080")
        options.add_argument(f"--user-agent={USER_AGENT}")
        self.timeout = timeout
        self.driver = webdriver.Chrome(options=options)
        self.driver.set_page_load_timeout(timeout)

    def fetch(self, url: str) -> dict[str, str | None]:
        self.driver.get(url)
        WebDriverWait(self.driver, self.timeout).until(
            lambda driver: driver.execute_script("return document.readyState") == "complete"
        )
        html = self.driver.page_source
        return {
            "url": url,
            "final_url": self.driver.current_url,
            "html": html,
            # Selenium không cung cấp HTTP status trực tiếp trong flow này.
            "http_status": None,
        }

    def close(self) -> None:
        self.driver.quit()

In [5]:
class VnExpressParser:
    """Source-specific selectors for the Step 1 VnExpress smoke test."""

    CONTENT_SELECTORS = (
        ".fck_detail p",
        "article p",
        ".content_detail p",
    )

    @staticmethod
    def _first_text(soup: BeautifulSoup, selectors: tuple[str, ...]) -> str | None:
        for selector in selectors:
            node = soup.select_one(selector)
            if node:
                text = clean_text(node.get_text(" ", strip=True))
                if text:
                    return text
        return None

    def parse(self, html: str) -> dict[str, str | None]:
        soup = BeautifulSoup(html, "lxml")

        title = self._first_text(soup, ("h1.title-detail", "h1", "title"))
        author = self._first_text(soup, (".author_mail", ".author"))

        published_at = None
        published_meta = soup.select_one("meta[property='article:published_time']")
        if published_meta:
            published_at = clean_text(published_meta.get("content"))
        if not published_at:
            published_at = self._first_text(soup, ("time[datetime]", ".date"))

        paragraphs_by_selector = [
            soup.select(selector) for selector in self.CONTENT_SELECTORS
        ]
        paragraphs = max(paragraphs_by_selector, key=len, default=[])
        content_parts = [
            text
            for paragraph in paragraphs
            if (text := clean_text(paragraph.get_text(" ", strip=True)))
        ]
        content = "\n\n".join(content_parts) or None

        image_meta = soup.select_one("meta[property='og:image']")
        thumbnail_url = image_meta.get("content") if image_meta else None

        return {
            "title": title,
            "author": author,
            "published_at": published_at,
            "content": content,
            "thumbnail_url": clean_text(thumbnail_url),
        }

In [6]:
@dataclass
class RawArticle:
    source: str
    url: str
    final_url: str | None
    title: str | None
    author: str | None
    published_at: str | None
    content: str | None
    thumbnail_url: str | None
    raw_html_path: str | None
    http_status: int | None
    crawl_status: str
    fetched_at: str
    error: str | None = None


def save_artifacts(html: str, article: RawArticle) -> RawArticle:
    fetched_at = datetime.fromisoformat(article.fetched_at)
    date_path = fetched_at.astimezone().strftime("%Y/%m/%d")
    article_id = url_hash(article.url)

    raw_path = OUTPUT_ROOT / "raw" / article.source / date_path / f"{article_id}.html.gz"
    metadata_path = OUTPUT_ROOT / "metadata" / article.source / date_path / f"{article_id}.json"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    with gzip.open(raw_path, "wt", encoding="utf-8") as file:
        file.write(html)

    article.raw_html_path = raw_path.relative_to(PROJECT_ROOT).as_posix()
    metadata_path.write_text(
        json.dumps(asdict(article), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return article


def crawl_article(url: str, fetcher: SeleniumFetcher, parser: VnExpressParser) -> RawArticle:
    normalized_url = normalize_url(url)
    fetched_at = datetime.now(timezone.utc).isoformat()
    html = ""
    response: dict[str, str | None] = {}

    try:
        response = fetcher.fetch(normalized_url)
        html = response.get("html") or ""
        if not html.strip():
            return RawArticle(
                source="vnexpress", url=normalized_url, final_url=response.get("final_url"),
                title=None, author=None, published_at=None, content=None, thumbnail_url=None,
                raw_html_path=None, http_status=None, crawl_status="EMPTY_HTML",
                fetched_at=fetched_at, error="Rendered HTML is empty",
            )

        parsed = parser.parse(html)
        status = "SUCCESS" if parsed.get("content") else "CONTENT_NOT_FOUND"
        article = RawArticle(
            source="vnexpress", url=normalized_url, final_url=response.get("final_url"),
            **parsed, raw_html_path=None, http_status=None, crawl_status=status,
            fetched_at=fetched_at, error=None if status == "SUCCESS" else "No article body matched",
        )
        return save_artifacts(html, article)

    except TimeoutException as error:
        status = "TIMEOUT"
    except WebDriverException as error:
        status = "BLOCKED"
    except Exception as error:
        status = "UNKNOWN_ERROR"

    article = RawArticle(
        source="vnexpress", url=normalized_url, final_url=response.get("final_url"),
        title=None, author=None, published_at=None, content=None, thumbnail_url=None,
        raw_html_path=None, http_status=None, crawl_status=status, fetched_at=fetched_at,
        error=str(error),
    )
    return save_artifacts(html, article) if html.strip() else article

In [7]:
def discover_article_urls(
    home_url: str,
    fetcher: SeleniumFetcher,
    limit: int = MAX_DISCOVERED_URLS,
) -> list[str]:
    response = fetcher.fetch(home_url)
    soup = BeautifulSoup(response.get("html") or "", "lxml")
    discovered: list[str] = []

    for anchor in soup.select("a[href]"):
        candidate = normalize_url(urljoin(home_url, anchor.get("href", "")))
        parts = urlsplit(candidate)
        is_vnexpress = parts.netloc == "vnexpress.net" or parts.netloc.endswith(".vnexpress.net")
        is_article = parts.path.endswith(".html")
        if is_vnexpress and is_article and candidate not in discovered:
            discovered.append(candidate)
        if len(discovered) >= limit:
            break

    return discovered

In [11]:
fetcher = SeleniumFetcher()
parser = VnExpressParser()
records: list[RawArticle] = []

try:
    try:
        if TARGET_ARTICLE_URL.strip():
            article_urls = [normalize_url(TARGET_ARTICLE_URL)]
        else:
            article_urls = discover_article_urls(HOME_URL)[:MAX_ARTICLES_TO_CRAWL]
    except Exception as error:
        article_urls = []
        print(f"Discovery failed: {error}")

    if not article_urls:
        print("Không tìm thấy URL bài viết. Hãy điền TARGET_ARTICLE_URL rồi chạy lại cell này.")

    print(f"URLs selected: {len(article_urls)}")
    for index, article_url in enumerate(article_urls, start=1):
        record = crawl_article(article_url, fetcher, parser)
        records.append(record)
        print(f"[{index}] {record.crawl_status}: {record.title or record.url}")
        if index < len(article_urls):
            time.sleep(2)
finally:
    fetcher.close()

URLs selected: 1
[1] SUCCESS: Áp thấp nhiệt đới mạnh lên thành bão


In [12]:
summary = [
    {
        "status": record.crawl_status,
        "title": record.title,
        "content_chars": len(record.content or ""),
        "raw_html_path": record.raw_html_path,
        "metadata_fields": [key for key, value in asdict(record).items() if value is not None],
    }
    for record in records
]
print(json.dumps(summary, ensure_ascii=False, indent=2))

[
  {
    "status": "SUCCESS",
    "title": "Áp thấp nhiệt đới mạnh lên thành bão",
    "content_chars": 3409,
    "raw_html_path": "crawl_data/raw/vnexpress/2026/08/21/b02a76741333967d.html.gz",
    "metadata_fields": [
      "source",
      "url",
      "final_url",
      "title",
      "published_at",
      "content",
      "thumbnail_url",
      "raw_html_path",
      "crawl_status",
      "fetched_at"
    ]
  }
]
